In [ ]:
import arcpy
import os

In [2]:
# 1. Create the new mosaic dataset that the .tif rasters will be added to 
# --------------------------------------------------------------------------------------------------------------

# Output geodatabase for new mosaic to be created in
gdb = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb"

# Mosaic dataset name
md_name = "RAW_GEBCO_MOSAIC_g4326"

# Mosaic coordinate system: GCS_WGS_1984 (EPSG:4326)
sr = arcpy.SpatialReference(4326)   #𖠜 SET TO MATCH COORDINATE SYSTEM OF INPUT .TIF FILES!

# 'Create Mosaic Dataset' tool (Product Definition: None (default))
arcpy.management.CreateMosaicDataset(
    gdb,
    md_name,
    sr
)
print(f"created mosaic dataset {md_name}")

created mosaic dataset RAW_GEBCO_MOSAIC_g4326


In [ ]:
# 2. Add all GEBCO downloaded .tif files to new mosaic "RAW_GEBCO_MOSAIC_g4326"
# --------------------------------------------------------------------------------------------------------------

# Path to mosaic dataset that rasters (.tif files) will be added to
md = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\RAW_GEBCO_MOSAIC_g4326"

# Path to root folder holding all raw GEBCO download subfolders with each individual .tif
root = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\raw_data\raw_gebco_tif_downloads_g4326"

# Use only the depth grids .tif files (actual bathymetry data), skip TID (type identifier) .tif files
tifs = []
for dirpath, _, files in os.walk(root):
    for f in files:
        if f.lower().endswith(".tif") and "_tid_" not in f.lower():
            tifs.append(os.path.join(dirpath, f))

In [6]:
# 'Add Rasters to Mosaic Dataset' tool
arcpy.management.AddRastersToMosaicDataset(
    md,
    "Raster Dataset",
    tifs,
    update_cellsize_ranges="UPDATE_CELL_SIZES",
    update_boundary="UPDATE_BOUNDARY",
    update_overviews="NO_OVERVIEWS",
    build_pyramids="BUILD_PYRAMIDS",
    calculate_statistics="CALCULATE_STATISTICS",
    duplicate_items_action="OVERWRITE_DUPLICATES" # matches on raster path so only overwrites if re-adding exact same file paths
)
print(f"added {len(tifs)} rasters")

added 12 rasters


In [ ]:
# 3. Export the mosaic to single raster with land removed
# --------------------------------------------------------------------------------------------------------------

# Check out Spatial Analyst (required for SetNull)
from arcpy.sa import SetNull, Raster
arcpy.CheckOutExtension("Spatial")

# Output raster in same gdb as mosaic
out_raster = gdb + r"\GBC15a_bathy_entireEastCarib_g4326"

# Create single standalone raster from mosaic dataset
r = Raster(md)
# 'SetNull' sets a cell to NoData where the condition is TRUE = null every cell with a value above 0 (land) and keep the rest
ocean = SetNull(r > 0, r)
ocean.save(out_raster)

# release Spatial Analyst license
arcpy.CheckInExtension("Spatial")

# Verify land was removed (MAXIMUM should now be <=0)
chk = arcpy.Raster(out_raster)
print(f"exported mosaic to {out_raster}")
print(f"MINIMUM: {chk.minimum}")
print(f"MAXIMUM: {chk.maximum}")

exported mosaic to C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\GBC15a_bathy_entireEastCarib_g4326
MINIMUM: -8733.0
MAXIMUM: 0.0
